<a href="https://colab.research.google.com/github/rahu2004/pyspark_ETL_RDD/blob/main/PySpark_ETL_RDD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel

In [3]:
spark = SparkSession.builder \
    .appName("OnlineRetailAnalytics") \
    .getOrCreate()

sc = spark.sparkContext

In [4]:
from google.colab import files
uploaded = files.upload()

Saving OnlineRetail.csv to OnlineRetail.csv


In [6]:
retail_df = spark.read.csv(
    '/content/OnlineRetail.csv',
    header=True,
    inferSchema=True
)

In [7]:
retail_df.show(5)
retail_df.printSchema()

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
only showing top 5 rows
root
 |-- InvoiceNo: string (nullable = true)
 |-

RDD


In [8]:
retail_rdd = sc.textFile('/content/OnlineRetail.csv')

In [9]:
uk_rdd = retail_rdd.filter(
    lambda x: 'United Kingdom' in x
)

In [10]:
print("UK Orders Count:", uk_rdd.count())

UK Orders Count: 495478


In [11]:
mapped_rdd = uk_rdd.map(
    lambda x: x.split(',')
)

In [12]:
mapped_rdd.take(5)

[['536365',
  '85123A',
  'WHITE HANGING HEART T-LIGHT HOLDER',
  '6',
  '12/1/2010 8:26',
  '2.55',
  '17850',
  'United Kingdom'],
 ['536365',
  '71053',
  'WHITE METAL LANTERN',
  '6',
  '12/1/2010 8:26',
  '3.39',
  '17850',
  'United Kingdom'],
 ['536365',
  '84406B',
  'CREAM CUPID HEARTS COAT HANGER',
  '8',
  '12/1/2010 8:26',
  '2.75',
  '17850',
  'United Kingdom'],
 ['536365',
  '84029G',
  'KNITTED UNION FLAG HOT WATER BOTTLE',
  '6',
  '12/1/2010 8:26',
  '3.39',
  '17850',
  'United Kingdom'],
 ['536365',
  '84029E',
  'RED WOOLLY HOTTIE WHITE HEART.',
  '6',
  '12/1/2010 8:26',
  '3.39',
  '17850',
  'United Kingdom']]

ETL

DUPLICATES ADN REMOVE NULL VALUES


In [13]:
retail_df = retail_df.dropna()
retail_df = retail_df.distinct()


Create Total Amount Column

In [15]:
retail_df = retail_df.withColumn(
    'TotalAmount',
    col('Quantity') * col('UnitPrice')
)
retail_df.show(5)

+---------+---------+--------------------+--------+---------------+---------+----------+--------------+------------------+
|InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|       TotalAmount|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+------------------+
|   536381|    21731|RED TOADSTOOL LED...|       2| 12/1/2010 9:41|     1.65|     15311|United Kingdom|               3.3|
|   536390|    22174|          PHOTO CUBE|      48|12/1/2010 10:19|     1.48|     17511|United Kingdom| 71.03999999999999|
|   536396|    82483|WOOD 2 DRAWER CAB...|       2|12/1/2010 10:51|     4.95|     17850|United Kingdom|               9.9|
|   536408|    22914|BLUE COAT RACK PA...|       3|12/1/2010 11:41|     4.95|     14307|United Kingdom|14.850000000000001|
|   536412|    22382|LUNCH BAG SPACEBO...|       3|12/1/2010 11:49|     1.65|     17920|United Kingdom| 4.949999999999999|
+---------+-----

In [16]:
retail_df.select(
    'InvoiceNo',
    'Description',
    'Quantity',
    'UnitPrice',
    'Country'
).show(5)

+---------+--------------------+--------+---------+--------------+
|InvoiceNo|         Description|Quantity|UnitPrice|       Country|
+---------+--------------------+--------+---------+--------------+
|   536381|RED TOADSTOOL LED...|       2|     1.65|United Kingdom|
|   536390|          PHOTO CUBE|      48|     1.48|United Kingdom|
|   536396|WOOD 2 DRAWER CAB...|       2|     4.95|United Kingdom|
|   536408|BLUE COAT RACK PA...|       3|     4.95|United Kingdom|
|   536412|LUNCH BAG SPACEBO...|       3|     1.65|United Kingdom|
+---------+--------------------+--------+---------+--------------+
only showing top 5 rows


FILTER HIGH VALUE PRODUCTS AND SORT

In [19]:
high_value_df = retail_df.filter(
    col('TotalAmount') > 1000
)

high_value_df.show(5)

retail_df.orderBy(
    col('TotalAmount').desc()
).show(10)

+---------+---------+--------------------+--------+---------------+---------+----------+--------------+-----------+
|InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|TotalAmount|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+-----------+
|   562439|    23215|JINGLE BELL HEART...|     576| 8/4/2011 18:06|     1.79|     12931|United Kingdom|    1031.04|
|   562439|    22197|      POPCORN HOLDER|    1900| 8/4/2011 18:06|     0.72|     12931|United Kingdom|     1368.0|
|   540689|    22469|HEART OF WICKER S...|    1356| 1/11/2011 8:43|     1.93|     17450|United Kingdom|    2617.08|
|   543379|    22189|CREAM HEART CARD ...|     504| 2/7/2011 15:37|     2.31|     18102|United Kingdom|    1164.24|
|   552904|    71477|COLOUR GLASS. STA...|     428|5/12/2011 11:07|     2.75|     16013|United Kingdom|     1177.0|
+---------+---------+--------------------+--------+---------------+-----

In [20]:
retail_df.agg(
    sum('TotalAmount').alias('TotalRevenue')
).show()

+-----------------+
|     TotalRevenue|
+-----------------+
|8278519.424000363|
+-----------------+



In [21]:
retail_df.agg(
    avg('TotalAmount').alias('AverageTransaction')
).show()

+------------------+
|AverageTransaction|
+------------------+
| 20.61363787213365|
+------------------+



In [22]:
print("Total Orders:", retail_df.count())

Total Orders: 401604


REVENUE BY COUNTRY

In [23]:
country_revenue = retail_df.groupBy(
    'Country'
).agg(
    sum('TotalAmount').alias('Revenue')
)

country_revenue.show(10)

+------------------+------------------+
|           Country|           Revenue|
+------------------+------------------+
|            Sweden| 36585.40999999999|
|         Singapore| 9120.390000000005|
|           Germany|221509.47000000003|
|               RSA|           1002.31|
|            France|196626.05000000022|
|            Greece| 4710.519999999999|
|European Community|           1291.75|
|           Belgium| 40910.95999999998|
|           Finland|22326.740000000005|
|             Malta|2505.4699999999993|
+------------------+------------------+
only showing top 10 rows


TOP PRODUCTS

In [24]:
product_sales = retail_df.groupBy(
    'Description'
).agg(
    sum('Quantity').alias('TotalQuantitySold')
)

product_sales.orderBy(
    col('TotalQuantitySold').desc()
).show(10)

+--------------------+-----------------+
|         Description|TotalQuantitySold|
+--------------------+-----------------+
|WORLD WAR 2 GLIDE...|            53119|
|JUMBO BAG RED RET...|            44963|
|ASSORTED COLOUR B...|            35215|
|WHITE HANGING HEA...|            34128|
|PACK OF 72 RETROS...|            33386|
|      POPCORN HOLDER|            30492|
|  RABBIT NIGHT LIGHT|            27045|
|MINI PAINT SET VI...|            25880|
|PACK OF 12 LONDON...|            25305|
|PACK OF 60 PINK P...|            24129|
+--------------------+-----------------+
only showing top 10 rows


Customer Revenue-WINDOW FUNCTIONS




In [28]:
customer_revenue = retail_df.groupBy(
    'CustomerID'
).agg(
    sum('TotalAmount').alias('CustomerSpending')
).show()

+----------+------------------+
|CustomerID|  CustomerSpending|
+----------+------------------+
|     15727| 5159.060000000003|
|     18024|236.77999999999992|
|     16339| 94.05000000000001|
|     17389|31300.079999999998|
|     13285| 2709.120000000001|
|     13623|             652.4|
|     14570|218.06000000000003|
|     16503|           1421.43|
|     14450|            483.25|
|     17420| 598.8300000000002|
|     16861|            151.65|
|     16386|            302.57|
|     15447|            155.17|
|     16574|            451.44|
|     12940|            862.44|
|     15957|428.89000000000004|
|     15790|            218.75|
|     17679|1992.1100000000001|
|     13832|40.949999999999996|
|     15619|336.40000000000003|
+----------+------------------+
only showing top 20 rows


In [26]:
window_spec = Window.orderBy(
    col('CustomerSpending').desc()
)

ranked_customers = customer_revenue.withColumn(
    'Rank',
    rank().over(window_spec)
)

ranked_customers.show(10)

+----------+------------------+----+
|CustomerID|  CustomerSpending|Rank|
+----------+------------------+----+
|     14646|279489.01999999955|   1|
|     18102|         256438.49|   2|
|     17450|187322.16999999998|   3|
|     14911|132458.72999999986|   4|
|     12415|         123725.45|   5|
|     14156|113214.59000000004|   6|
|     17511| 88125.37999999999|   7|
|     16684|          65892.08|   8|
|     13694|          62690.54|   9|
|     15311| 59284.19000000009|  10|
+----------+------------------+----+
only showing top 10 rows


spark sql

In [29]:
retail_df.createOrReplaceTempView('retail')

In [30]:
spark.sql("""
SELECT Country,
       SUM(TotalAmount) AS Revenue
FROM retail
GROUP BY Country
ORDER BY Revenue DESC
""").show()

+---------------+------------------+
|        Country|           Revenue|
+---------------+------------------+
| United Kingdom| 6747156.154000033|
|    Netherlands|284661.53999999946|
|           EIRE|250001.77999999956|
|        Germany|221509.47000000003|
|         France|196626.05000000022|
|      Australia|         137009.77|
|    Switzerland| 55739.40000000011|
|          Spain| 54756.03000000022|
|        Belgium| 40910.95999999998|
|         Sweden| 36585.40999999999|
|          Japan| 35340.62000000002|
|         Norway|35163.460000000014|
|       Portugal|28995.759999999966|
|        Finland|22326.740000000005|
|Channel Islands| 20076.38999999999|
|        Denmark|18768.139999999985|
|          Italy|16890.510000000002|
|         Cyprus|12858.760000000002|
|        Austria|10154.320000000005|
|      Singapore| 9120.390000000005|
+---------------+------------------+
only showing top 20 rows


In [31]:
spark.sql("""
SELECT CustomerID,
       SUM(TotalAmount) AS Spending
FROM retail
GROUP BY CustomerID
ORDER BY Spending DESC
""").show()

+----------+------------------+
|CustomerID|          Spending|
+----------+------------------+
|     14646|279489.01999999955|
|     18102|         256438.49|
|     17450|187322.16999999998|
|     14911|132458.72999999986|
|     12415|         123725.45|
|     14156|113214.59000000004|
|     17511| 88125.37999999999|
|     16684|          65892.08|
|     13694|          62690.54|
|     15311| 59284.19000000009|
|     13089| 57322.13000000008|
|     14096| 57120.91000000006|
|     15061|54228.739999999976|
|     16029| 53168.68999999999|
|     17949|          52750.84|
|     15769|          51823.72|
|     14298| 50862.43999999992|
|     14088| 50415.48999999995|
|     17841|39869.050000000236|
|     13798| 36352.86999999998|
+----------+------------------+
only showing top 20 rows


In [32]:
sample_df = retail_df.sample(
    withReplacement=False,
    fraction=0.05,
    seed=42
)

sample_df.show(5)

+---------+---------+--------------------+--------+----------------+---------+----------+--------------+-----------------+
|InvoiceNo|StockCode|         Description|Quantity|     InvoiceDate|UnitPrice|CustomerID|       Country|      TotalAmount|
+---------+---------+--------------------+--------+----------------+---------+----------+--------------+-----------------+
|   536793|    21912|VINTAGE SNAKES & ...|       4| 12/2/2010 15:39|     3.75|     16203|United Kingdom|             15.0|
|   538199|    22530|MAGIC DRAWING SLA...|       5|12/10/2010 11:02|     0.42|     14082|United Kingdom|              2.1|
|   538629|    22751|FELTCRAFT PRINCES...|       3|12/13/2010 13:10|     3.75|     17799|United Kingdom|            11.25|
|   539044|    22878|NUMBER TILE COTTA...|      10|12/15/2010 15:47|      2.1|     15727|United Kingdom|             21.0|
|   539215|    22082|RIBBON REEL STRIP...|       3|12/16/2010 12:42|     1.65|     15574|United Kingdom|4.949999999999999|
+---------+-----

partition

In [33]:
print(
    retail_df.rdd.getNumPartitions()
)

2


In [34]:
repartitioned_df = retail_df.repartition(4)

print(
    repartitioned_df.rdd.getNumPartitions()
)

4


coalesce

In [35]:
coalesced_df = repartitioned_df.coalesce(2)

print(
    coalesced_df.rdd.getNumPartitions()
)

2


In [36]:
retail_df.cache()

DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: string, UnitPrice: double, CustomerID: int, Country: string, TotalAmount: double]

persistance

In [37]:
retail_df.persist(
    StorageLevel.MEMORY_AND_DISK
)

DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: string, UnitPrice: double, CustomerID: int, Country: string, TotalAmount: double]

In [38]:
discount = sc.broadcast(0.10)

In [39]:
retail_df = retail_df.withColumn(
    'DiscountPrice',
    col('TotalAmount') - (col('TotalAmount') * discount.value)
)

retail_df.show(5)

+---------+---------+--------------------+--------+---------------+---------+----------+--------------+------------------+------------------+
|InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|       TotalAmount|     DiscountPrice|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+------------------+------------------+
|   536381|    21731|RED TOADSTOOL LED...|       2| 12/1/2010 9:41|     1.65|     15311|United Kingdom|               3.3|2.9699999999999998|
|   536390|    22174|          PHOTO CUBE|      48|12/1/2010 10:19|     1.48|     17511|United Kingdom| 71.03999999999999| 63.93599999999999|
|   536396|    82483|WOOD 2 DRAWER CAB...|       2|12/1/2010 10:51|     4.95|     17850|United Kingdom|               9.9|              8.91|
|   536408|    22914|BLUE COAT RACK PA...|       3|12/1/2010 11:41|     4.95|     14307|United Kingdom|14.850000000000001|13.365000000000002|
|   53

In [40]:
retail_df.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(DiscountPrice, '`-`('TotalAmount, '`*`('TotalAmount, 0.1)), None)]
+- Project [InvoiceNo#34, StockCode#35, Description#36, Quantity#37, InvoiceDate#38, UnitPrice#39, CustomerID#40, Country#41, (cast(Quantity#37 as double) * UnitPrice#39) AS TotalAmount#77]
   +- Project [InvoiceNo#34, StockCode#35, Description#36, Quantity#37, InvoiceDate#38, UnitPrice#39, CustomerID#40, Country#41, (cast(Quantity#37 as double) * UnitPrice#39) AS TotalAmount#76]
      +- Deduplicate [Quantity#37, UnitPrice#39, StockCode#35, Description#36, InvoiceDate#38, InvoiceNo#34, Country#41, CustomerID#40]
         +- Filter atleastnnonnulls(8, InvoiceNo#34, StockCode#35, Description#36, Quantity#37, InvoiceDate#38, UnitPrice#39, CustomerID#40, Country#41)
            +- Relation [InvoiceNo#34,StockCode#35,Description#36,Quantity#37,InvoiceDate#38,UnitPrice#39,CustomerID#40,Country#41] csv

== Analyzed Logical Plan ==
InvoiceNo: string, StockCode: stri

In [42]:
retail_df = retail_df.withColumn(
    "InvoiceDate",
    to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm")
)

monthly_revenue = retail_df.groupBy(
    month("InvoiceDate").alias("Month")
).agg(
    sum("TotalAmount").alias("Revenue")
)

monthly_revenue.orderBy("Month").show()

+-----+------------------+
|Month|           Revenue|
+-----+------------------+
|    1|473731.89999999985|
|    2|435534.06999999995|
|    3| 578576.2099999995|
|    4|425222.67100000015|
|    5|         647011.67|
|    6| 606862.5200000003|
|    7| 573112.3209999999|
|    8| 615078.0900000001|
|    9| 929356.2319999991|
|   10| 973306.3799999994|
|   11|1126815.0699999998|
|   12| 893912.2900000002|
+-----+------------------+



In [43]:
ranked_customers.orderBy(
    col('CustomerSpending').desc()
).show(10)

+----------+------------------+----+
|CustomerID|  CustomerSpending|Rank|
+----------+------------------+----+
|     14646|         279489.02|   1|
|     18102|         256438.49|   2|
|     17450|187322.17000000013|   3|
|     14911|         132458.73|   4|
|     12415|123725.45000000001|   5|
|     14156|113214.59000000003|   6|
|     17511| 88125.38000000003|   7|
|     16684| 65892.08000000002|   8|
|     13694| 62690.54000000001|   9|
|     15311| 59284.18999999997|  10|
+----------+------------------+----+
only showing top 10 rows


SAVE RESULTS


In [44]:
retail_df.write.csv(
    'retail_output',
    header=True,
    mode='overwrite'
)

In [45]:
spark.stop()